# 03. 라벨 수정 + 애매한 칸 삭제 (CPU, 2분)

01·02의 판정표로 Train·Val 라벨을 고친다. **Train과 Val에 같은 규칙**을 쓴다. 대화는 지우지 않고 그 증상 칸만 처리한다.

| 원래 라벨 | Qwen 확률 | 처리 |
|---|---|---|
| 있음 | p_호소 ≥ 0.5 | 그대로 |
| 있음 | p_없음 + p_부정 ≥ 0.9 | **'없음'으로 수정** |
| 있음 | 그 사이 (대원만 물음 등) | 삭제 (학습·채점에서 그 칸만 뺌) |
| 없음 | p_호소 ≥ 0.9 | **'있음'으로 수정** |
| 없음 | 0.5 ≤ p_호소 < 0.9 | 삭제 |
| 없음 | p_호소 < 0.5 | 그대로 |

삭제한 칸은 마스크 0이 되어 loss와 채점에서 빠진다(`trainer.masked_bce`, `metrics.f1_report`).

| 단계 | 하는 일 | 쓰는 함수 |
|---|---|---|
| 0 | 준비 | |
| 1 | (있으면) 검수 결과 정확도 보기 | `cleaning.review_accuracy` |
| 2 | 증상별 처리 방식 정하기 (수정 / 삭제만 / 안 함) | |
| 3 | 처리 표시 + **증상별 분포표·경고** | `cleaning.decide_actions`, `cleaning.action_summary` |
| 4 | 고친 라벨·마스크 저장 | `cleaning.build_fixed`, `cleaning.save_fixed` |

결과: `data/train_fixed.npz`, `data/val_fixed.npz`, `results/03_fix/`. 원본 라벨 파일은 건드리지 않는다.

## 0. 준비

In [ ]:
# 드라이브 연결 + src 폴더를 import할 수 있게 경로 추가
from google.colab import drive
drive.mount("/content/drive")

import os, sys
PROJECT_ROOT = "/content/drive/MyDrive/증상인식_최종_프로젝트"   # 폴더를 다른 곳에 두었으면 여기만 바꾸기
os.environ["SYMPTOM_FINAL_ROOT"] = PROJECT_ROOT
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
# src/*.py를 고친 뒤에는 런타임 → 세션 다시 시작 후 이 셀부터 다시 실행

In [ ]:
import pandas as pd
from src import config
from src.data import load_labels, prep
from src.cells import load_judged
from src.cleaning import decide_actions, action_summary, build_fixed, save_fixed, review_accuracy

jcfg = config.JudgeConfig()     # 기준값을 바꾸려면 config.JudgeConfig(thr_keep=..., thr_fix=...)
OUT = f"{config.RESULTS_DIR}/03_fix"
os.makedirs(OUT, exist_ok=True)
os.makedirs(config.DATA_DIR, exist_ok=True)

## 1. 검수 결과 정확도 (있으면)
01의 `검수_val.xlsx`에 O/X를 채워 같은 폴더에 `검수_val_완료.xlsx`로 저장해 두면, 처리 종류 × 증상별 정확도를 보여 준다.

In [ ]:
done_path = f"{config.RESULTS_DIR}/01_val/검수_val_완료.xlsx"
if os.path.exists(done_path):
    display(review_accuracy(pd.read_excel(done_path)))
else:
    print("검수 완료 파일 없음 → 건너뜀")

## 2. 증상별 처리 방식
- `"수정"`: 위 표대로 (기본)
- `"삭제만"`: 수정할 칸도 삭제로 돌린다. 수정 정확도가 0.9 미만인 증상
- `"안 함"`: 원래 라벨 그대로. 삭제까지 못 믿을 증상

In [ ]:
MODES = {s: "수정" for s in config.LABELS}
# 예: MODES["전신쇠약"] = "삭제만"
print(MODES)
print(f"기준: 그대로 {jcfg.thr_keep}, 수정 {jcfg.thr_fix}")

## 3. 처리 표시 + 증상별 분포표
**최종 있음** = 원래 있음 - 있음→없음 수정 - 있음 칸 삭제 + 없음→있음 수정.
'경고' 열이 비어 있지 않은 증상은 정리 후 '있음'이 너무 적다(Train 500개, Val 60개 미만 또는 원래의 50% 미만).
경고가 나오면 그 증상을 '삭제만'이나 '안 함'으로 바꾸고 다시 실행하는 것을 검토한다.

In [ ]:
tr_df, va_df = load_labels()
tr_j = decide_actions(load_judged(f"{config.JUDGE_DIR}/train_judged.csv"), jcfg, MODES)
va_j = decide_actions(load_judged(f"{config.JUDGE_DIR}/val_judged.csv"), jcfg, MODES)
s_tr = action_summary(tr_j, tr_df, jcfg.warn_min_pos_train, jcfg)
s_va = action_summary(va_j, va_df, jcfg.warn_min_pos_val, jcfg)
both = pd.concat({"Train": s_tr, "Val": s_va}, axis=1)
both.to_csv(f"{OUT}/처리_분포표.csv", encoding="utf-8-sig")
print("[Train]"); display(s_tr)
print("[Val]");   display(s_va)
warn = [f"{n} {s}: {w}" for n, t in [("Train", s_tr), ("Val", s_va)] for s, w in t["경고"].items() if w]
print("\n경고:", *(warn or ["없음"]), sep="\n- ")

## 4. 고친 라벨·마스크 저장
04와 같은 행 순서를 쓰도록 `data.prep`을 똑같이 거친다(Train은 발화 없는 대화 제외, Val은 전부).

In [ ]:
tr = prep(tr_df, "[SEP]", drop_empty=True)
va = prep(va_df, "[SEP]", drop_empty=False)
print("Train"); Y_tr, M_tr = build_fixed(tr, tr_j)
print("Val");   Y_va, M_va = build_fixed(va, va_j)
save_fixed(f"{config.DATA_DIR}/train_fixed.npz", tr, Y_tr, M_tr)
save_fixed(f"{config.DATA_DIR}/val_fixed.npz", va, Y_va, M_va)
tr_j.to_csv(f"{OUT}/train_처리표.csv", index=False, encoding="utf-8-sig")
va_j.to_csv(f"{OUT}/val_처리표.csv", index=False, encoding="utf-8-sig")